# Automated District Rainfall Collection

### This notebook obtains representative coordinates for each project district, downloads NASA POWER daily precipitation data, aggregates it into yearly  rainfall totals, and prepares a district-level rainfall dataset.

In [1]:
import pandas as pd
import numpy as np
import requests
import time

rainfall_daily_all = pd.read_csv(
    "../../data/processed/rainfall_daily_2021_2024.csv"
)

rainfall_daily_all["Date"] = pd.to_datetime(
    rainfall_daily_all["Date"]
)




In [2]:
production_history = pd.read_csv("../../data/processed/production_history_2021_2025.csv")

In [3]:
production_history.head()

,State,District,Crop,Year,Area,Production,Yield,Year_Order,Years_ Available,Complete_4_year_history,Previous_Yield,Previous_yield_percentage,Previous_area,Previous_Area_Percentage,Previous_Production,Production_Change_Pct,History_avg_yield,Yield_Deviation_Pct
0,Haryana,Ambala,Maize,2021-22,0.00,0.01,3945.0,2021,4,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Haryana,Ambala,Maize,2022-23,0.00,0.00,3455.0,2022,4,True,3945.0,-12.420786,0.0,NaN,0.01,-100.0,3945.0,-12.420786
2,Haryana,Ambala,Maize,2023-24,0.00,0.01,4642.0,2023,4,True,3455.0,34.356006,0.0,NaN,0.00,NaN,3700.0,25.459459
3,Haryana,Ambala,Maize,2024-25,0.00,0.00,3714.0,2024,4,True,4642.0,-19.991383,0.0,NaN,0.01,-100.0,4014.0,-7.473842
4,Haryana,Ambala,Rice,2021-22,0.94,3.86,4107.0,2021,4,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Creating a unique district list 

districts = (
    production_history[
    ["State" , "District"]
    ].drop_duplicates().sort_values(["State" , "District"]).reset_index(drop=True))



In [5]:
districts.head()

,State,District
0,Haryana,Ambala
1,Haryana,Bhiwani
2,Haryana,Charkhi Dadri
3,Haryana,Faridabad
4,Haryana,Fatehabad


In [6]:
districts.shape

(120, 2)

In [7]:
# Checking the district count

districts.groupby("State")["District"].nunique()

State
Haryana          22
Punjab           23
Uttar Pradesh    75
Name: District, dtype: int64

### Need latitude and longitude for NASA Power

In [8]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [9]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [10]:
# Creating the geocoder

geolocator = Nominatim(
    user_agent="crop-yield-risk-dashboard-data-analytics"
)



In [11]:
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1)

In [12]:
# Testing the geocode for one district first 

location = geocode("Meerut district, Uttar Pradesh, India")

location

Location(Meerut, Uttar Pradesh, India, (29.0018557, 77.7679671, 0.0))

In [13]:
location.latitude

29.0018557

In [14]:
location.longitude

77.7679671

In [15]:
# Creating a geocoding function

def get_coordinates(row):
    
    query = (
        f"{row['District']} district, "
        f"{row['State']}, India"
    )

    try:
        location = geocode(query)

        if location:
            return pd.Series(
                [
                    location.latitude,
                    location.longitude,
                    location.address
                ]
            )

    except Exception as e:
        print(
            "Error:",
            row["District"],
            row["State"],
            e
        )

    return pd.Series(
        [np.nan, np.nan, None]
    )

  




In [16]:
# Geocoding all districts

districts[
    [
        "Latitude",
        "Longitude",
        "Geocoded_Address"
    ]
] = districts.apply(
    get_coordinates,
    axis=1
)

In [17]:
districts.isnull().sum()

State               0
District            0
Latitude            2
Longitude           2
Geocoded_Address    2
dtype: int64

In [18]:
failed_geocoding = districts[districts["Latitude"].isnull() | districts["Longitude"].isnull()]

failed_geocoding


,State,District,Latitude,Longitude,Geocoded_Address
42,Punjab,Shahid Bhagat Singh Nagar,NaN,NaN,NaN
113,Uttar Pradesh,Shrawasti,NaN,NaN,NaN


In [19]:
# Validate returned location 

districts[
["State" , "District" , "Geocoded_Address"]].head(20)

,State,District,Geocoded_Address
0,Haryana,Ambala,"Ambala, Haryana, India"
1,Haryana,Bhiwani,"Bhiwani, Haryana, India"
2,Haryana,Charkhi Dadri,"Charkhi Dadri, Haryana, India"
3,Haryana,Faridabad,"Faridabad, Haryana, India"
4,Haryana,Fatehabad,"Fatehabad, Haryana, India"
5,Haryana,Gurugram,"District And Sessions Court Gurugram, Sohna Ro..."
6,Haryana,Hisar,"Hisar, Haryana, India"
7,Haryana,Jhajjar,"Jhajjar, Haryana, India"
8,Haryana,Jind,"Jind, Haryana, India"
9,Haryana,Kaithal,"Kaithal, Haryana, India"


In [20]:
# Saving the coordinates

districts.to_csv(
    "../../data/processed/district_coordinates.csv",
    index=False
)

In [21]:
# Testing NASA API for district Meerut

NASA_URL = (
    "https://power.larc.nasa.gov/"
    "api/temporal/daily/point"
)

In [22]:
meerut = districts[
    (districts["State"] == "Uttar Pradesh")
    & (districts["District"] == "Meerut")
].iloc[0]

In [23]:
params = {
    "parameters": "PRECTOTCORR",
    "community": "AG",
    "longitude": meerut["Longitude"],
    "latitude": meerut["Latitude"],
    "start": "20210101",
    "end": "20241231",
    "format": "JSON"
}

In [24]:
response = requests.get(
    NASA_URL,
    params=params,
    timeout=60
)

In [25]:
response.status_code

200

In [26]:
# Inspecting NASA json 

data = response.json()

data.keys()

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])

In [27]:
data["properties"]["parameter"]["PRECTOTCORR"]

{'20210101': 0.0,
 '20210102': 5.02,
 '20210103': 15.6,
 '20210104': 0.22,
 '20210105': 6.13,
 '20210106': 7.16,
 '20210107': 0.1,
 '20210108': 0.21,
 '20210109': 0.71,
 '20210110': 0.0,
 '20210111': 0.0,
 '20210112': 0.0,
 '20210113': 0.0,
 '20210114': 0.0,
 '20210115': 0.0,
 '20210116': 0.0,
 '20210117': 0.0,
 '20210118': 0.0,
 '20210119': 0.0,
 '20210120': 0.0,
 '20210121': 0.0,
 '20210122': 0.0,
 '20210123': 0.0,
 '20210124': 0.0,
 '20210125': 0.0,
 '20210126': 0.0,
 '20210127': 0.0,
 '20210128': 0.0,
 '20210129': 0.0,
 '20210130': 0.0,
 '20210131': 0.0,
 '20210201': 0.0,
 '20210202': 0.0,
 '20210203': 0.18,
 '20210204': 4.95,
 '20210205': 0.23,
 '20210206': 0.0,
 '20210207': 0.0,
 '20210208': 0.0,
 '20210209': 0.0,
 '20210210': 0.0,
 '20210211': 0.0,
 '20210212': 0.0,
 '20210213': 0.0,
 '20210214': 0.0,
 '20210215': 0.0,
 '20210216': 0.0,
 '20210217': 0.0,
 '20210218': 0.0,
 '20210219': 0.0,
 '20210220': 0.0,
 '20210221': 0.0,
 '20210222': 0.0,
 '20210223': 0.0,
 '20210224': 0.0,


In [28]:
# Converting one NASA reponse into dataframe 

rainfall_dict = (  data["properties"]["parameter"]["PRECTOTCORR"]
)

In [29]:
test_rainfall = pd.DataFrame(rainfall_dict.items(),columns=["Date", "Rainfall_mm"])

In [30]:
test_rainfall.head()

,Date,Rainfall_mm
0,20210101,0.00
1,20210102,5.02
2,20210103,15.60
3,20210104,0.22
4,20210105,6.13


In [31]:
# Converting the date

test_rainfall["Date"] = pd.to_datetime(
    test_rainfall["Date"],
    format="%Y%m%d"
)

In [32]:
test_rainfall.head()

,Date,Rainfall_mm
0,2021-01-01,0.00
1,2021-01-02,5.02
2,2021-01-03,15.60
3,2021-01-04,0.22
4,2021-01-05,6.13


In [33]:
test_rainfall["Year"] = (
    test_rainfall["Date"].dt.year
)

In [34]:
test_rainfall.head()

,Date,Rainfall_mm,Year
0,2021-01-01,0.00,2021
1,2021-01-02,5.02,2021
2,2021-01-03,15.60,2021
3,2021-01-04,0.22,2021
4,2021-01-05,6.13,2021


In [35]:
# Aggregating test result to annual rainfall 

test_annual = (
    test_rainfall
    .groupby(
        "Year",
        as_index=False
    )["Rainfall_mm"]
    .sum()
)

test_annual

,Year,Rainfall_mm
0,2021,1124.21
1,2022,857.40
2,2023,1007.27
3,2024,927.64


In [36]:
# Creating the NASA function 
def fetch_nasa_rainfall(
    state,
    district,
    latitude,
    longitude,
    start_date,
    end_date
):
    params = {
        "parameters": "PRECTOTCORR",
        "community": "AG",
        "longitude": longitude,
        "latitude": latitude,
        "start": start_date,
        "end": end_date,
        "format": "JSON"
    }

    try:
        response = requests.get(
            NASA_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        rainfall_dict = (
            data["properties"]
                ["parameter"]
                ["PRECTOTCORR"]
        )

        temp = pd.DataFrame(
            rainfall_dict.items(),
            columns=[
                "Date",
                "Rainfall_mm"
            ]
        )

        temp["Date"] = pd.to_datetime(
            temp["Date"],
            format="%Y%m%d"
        )

        temp["Year"] = temp["Date"].dt.year

        temp["State"] = state
        temp["District"] = district
        temp["Latitude"] = latitude
        temp["Longitude"] = longitude

        return temp

    except Exception as e:
        print(
            f"Failed: {district}, {state}: {e}"
        )

        return None

In [37]:
# Fetch Jan–Jun 2025 for every district

rainfall_2025_part = []

In [38]:
for _, row in districts.dropna(
    subset=[
        "Latitude",
        "Longitude"
    ]
).iterrows():

    print(
        "Fetching 2025:",
        row["District"],
        row["State"]
    )

    result = fetch_nasa_rainfall(
        row["State"],
        row["District"],
        row["Latitude"],
        row["Longitude"],
        "20250101",
        "20250630"
    )

    if result is not None:
        rainfall_2025_part.append(result)

    time.sleep(1)

Fetching 2025: Ambala Haryana
Fetching 2025: Bhiwani Haryana
Fetching 2025: Charkhi Dadri Haryana
Fetching 2025: Faridabad Haryana
Fetching 2025: Fatehabad Haryana
Fetching 2025: Gurugram Haryana
Fetching 2025: Hisar Haryana
Fetching 2025: Jhajjar Haryana
Fetching 2025: Jind Haryana
Fetching 2025: Kaithal Haryana
Fetching 2025: Karnal Haryana
Fetching 2025: Kurukshetra Haryana
Fetching 2025: Mahendragarh Haryana
Fetching 2025: Nuh Haryana
Fetching 2025: Palwal Haryana
Fetching 2025: Panchkula Haryana
Fetching 2025: Panipat Haryana
Fetching 2025: Rewari Haryana
Fetching 2025: Rohtak Haryana
Fetching 2025: Sirsa Haryana
Fetching 2025: Sonipat Haryana
Fetching 2025: Yamunanagar Haryana
Fetching 2025: Amritsar Punjab
Fetching 2025: Barnala Punjab
Fetching 2025: Bathinda Punjab
Fetching 2025: Faridkot Punjab
Fetching 2025: Fatehgarh Sahib Punjab
Fetching 2025: Fazilka Punjab
Fetching 2025: Ferozepur Punjab
Fetching 2025: Gurdaspur Punjab
Fetching 2025: Hoshiarpur Punjab
Fetching 2025: Jalan

In [42]:
# Combine all rainfall

rainfall_daily_all = pd.concat(
    all_rainfall,
    ignore_index=True
)

rainfall_daily_all.head()

NameError: name 'all_rainfall' is not defined

In [43]:
rainfall_daily_all.shape

(175320, 7)

In [44]:
# Handling Nasa missing values

(rainfall_daily_all["Rainfall_mm"] == -999).sum()

np.int64(0)

In [45]:
rainfall_daily_all["Rainfall_mm"] = (
    rainfall_daily_all["Rainfall_mm"]
    .replace(-999, np.nan)
)

In [46]:
rainfall_daily_all[
    "Rainfall_mm"
].isnull().sum()

np.int64(0)

In [47]:
# Aggregating all districts annually

rainfall_history = (
    rainfall_daily_all
    .groupby(
        [
            "State",
            "District",
            "Year"
        ],
        as_index=False
    )
    .agg(
        Rainfall_mm=(
            "Rainfall_mm",
            "sum"
        ),
        Rainfall_Days_Available=(
            "Rainfall_mm",
            "count"
        )
    )
)

In [48]:
# Validating year completeness

pd.crosstab(
    rainfall_history["Year"],
    rainfall_history["State"]
)



State,Haryana,Punjab,Uttar Pradesh
Year,,,
2021,22,23,75
2022,22,23,75
2023,22,23,75
2024,22,23,75


In [49]:
rainfall_history[
    "Rainfall_Days_Available"
].value_counts().sort_index()

Rainfall_Days_Available
365    360
366    120
Name: count, dtype: int64

In [50]:
# Saving rainfall dataset

rainfall_daily_all.to_csv(
    "../../data/processed/rainfall_daily_2021_2024.csv",
    index=False
)

In [51]:
#Saving annual rainfall dataset

rainfall_history.to_csv(
    "../../data/processed/rainfall_history_2021_2024.csv",
    index=False
)

In [52]:
problem_districts = [
    "Shahid Bhagat Singh Nagar",
    "Shrawasti"
]

districts[
    districts["District"].isin(problem_districts)
][
    [
        "State",
        "District",
        "Latitude",
        "Longitude",
        "Geocoded_Address"
    ]
]

,State,District,Latitude,Longitude,Geocoded_Address
42,Punjab,Shahid Bhagat Singh Nagar,NaN,NaN,NaN
113,Uttar Pradesh,Shrawasti,NaN,NaN,NaN


In [53]:
# Manually geocding sharwasti

shrawasti_location = geocode(
    "Shravasti district, Uttar Pradesh, India"
)

shrawasti_location

Location(Sravasti, Ikauna, Sharavasti, Uttar Pradesh, India, (27.5083429, 82.0263569, 0.0))

In [54]:
print(shrawasti_location.latitude)
print(shrawasti_location.longitude)
print(shrawasti_location.address)

27.5083429
82.0263569
Sravasti, Ikauna, Sharavasti, Uttar Pradesh, India


In [55]:
mask = (
    (districts["State"] == "Uttar Pradesh")
    &
    (districts["District"] == "Shrawasti")
)

districts.loc[
    mask,
    "Latitude"
] = shrawasti_location.latitude

districts.loc[
    mask,
    "Longitude"
] = shrawasti_location.longitude

districts.loc[
    mask,
    "Geocoded_Address"
] = shrawasti_location.address

In [56]:
districts[
    (districts["State"] == "Uttar Pradesh")
    &
    (districts["District"] == "Shrawasti")
]

,State,District,Latitude,Longitude,Geocoded_Address
113,Uttar Pradesh,Shrawasti,27.508343,82.026357,"Sravasti, Ikauna, Sharavasti, Uttar Pradesh, I..."


In [57]:
# Fixing the other district 

sbs_location = geocode(
    "Nawanshahr, Punjab, India"
)

sbs_location

Location(Shaheed Bhagat Singh Nagar, Punjab, India, (31.126969, 76.1598988, 0.0))

In [58]:
print(sbs_location.latitude)
print(sbs_location.longitude)
print(sbs_location.address)

31.126969
76.1598988
Shaheed Bhagat Singh Nagar, Punjab, India


In [59]:
mask = (
    (districts["State"] == "Punjab")
    &
    (
        districts["District"]
        == "Shahid Bhagat Singh Nagar"
    )
)

districts.loc[
    mask,
    "Latitude"
] = sbs_location.latitude

districts.loc[
    mask,
    "Longitude"
] = sbs_location.longitude

districts.loc[
    mask,
    "Geocoded_Address"
] = sbs_location.address

In [60]:
districts[
    districts["District"]
    == "Shahid Bhagat Singh Nagar"
]

,State,District,Latitude,Longitude,Geocoded_Address
42,Punjab,Shahid Bhagat Singh Nagar,31.126969,76.159899,"Shaheed Bhagat Singh Nagar, Punjab, India"


In [61]:
# Saving corrected coordinates

districts.to_csv(
    "../../data/processed/district_coordinates.csv",
    index=False
)

In [62]:
problem_coordinates = districts[
    districts["District"].isin(
        [
            "Shahid Bhagat Singh Nagar",
            "Shrawasti"
        ]
    )
]

problem_coordinates

,State,District,Latitude,Longitude,Geocoded_Address
42,Punjab,Shahid Bhagat Singh Nagar,31.126969,76.159899,"Shaheed Bhagat Singh Nagar, Punjab, India"
113,Uttar Pradesh,Shrawasti,27.508343,82.026357,"Sravasti, Ikauna, Sharavasti, Uttar Pradesh, I..."


In [67]:
missing_district_rainfall = []

for _, row in problem_coordinates.iterrows():

    print(
        "Fetching:",
        row["District"],
        row["State"]
    )

    result = fetch_nasa_rainfall(
        row["State"],
        row["District"],
        row["Latitude"],
        row["Longitude"],
        "20210101",
        "20241231"
    )

    if result is not None:
        missing_district_rainfall.append(result)

Fetching: Shahid Bhagat Singh Nagar Punjab
Fetching: Shrawasti Uttar Pradesh


In [68]:
missing_district_rainfall = pd.concat(
    missing_district_rainfall,
    ignore_index=True
)

In [69]:
rainfall_daily_all = pd.concat(
    [
        rainfall_daily_all,
        missing_district_rainfall
    ],
    ignore_index=True
)

In [70]:
rainfall_daily_all = (
    rainfall_daily_all
    .drop_duplicates(
        subset=[
            "State",
            "District",
            "Date"
        ]
    )
    .reset_index(drop=True)
)

In [71]:
rainfall_daily_all.to_csv(
    "../../data/processed/"
    "rainfall_daily_2021_2024.csv",
    index=False
)

In [72]:
# Combinig 2025 rainfall

rainfall_2025_part = pd.concat(
    rainfall_2025_part,
    ignore_index=True
)

In [73]:
rainfall_daily_extended = pd.concat(
    [
        rainfall_daily_all,
        rainfall_2025_part
    ],
    ignore_index=True
)

In [74]:
rainfall_daily_extended = (
    rainfall_daily_extended
    .drop_duplicates(
        subset=[
            "State",
            "District",
            "Date"
        ]
    )
    .sort_values(
        [
            "State",
            "District",
            "Date"
        ]
    )
    .reset_index(drop=True)
)

In [75]:

# Verifying the data range
rainfall_daily_extended["Date"].min()
rainfall_daily_extended["Date"].max()

Timestamp('2025-06-30 00:00:00')

In [76]:
print("Start:", rainfall_daily_extended["Date"].min())
print("End:", rainfall_daily_extended["Date"].max())

Start: 2021-01-01 00:00:00
End: 2025-06-30 00:00:00


In [77]:
rainfall_daily_extended.duplicated(
    subset=["State", "District", "Date"]
).sum()

np.int64(0)

In [78]:
rainfall_daily_extended.to_csv(
    "../../data/processed/rainfall_daily_2021_2025.csv",
    index=False
)